
# 02 — Text Preprocessing and Feature Engineering

## 1. Dataset loading and preparation

This notebook transforms the titles and abstracts of 75,110 arXiv (cs.CL category) NLP publications (2021–2025) into numerical features for unsupervised clustering.

The analysis uses the filtered corpus produced in `01_data_acquisition.ipynb`. The initial preparation consists of loading the required metadata, normalising whitespace, and validating data integrity before text vectorisation.

In [2]:
import json
from pathlib import Path
import pandas as pd

# Path to the filtered corpus from Stage 1
data_path = Path("data/arxiv_cscl_2021_2025.jsonl")

if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

# Select only the fields needed for the analysis
columns = [
    "id",
    "title",
    "abstract",
    "categories",
    "first_submission_date"
]

# Read the filtered JSONL file sequentially
records = []

with data_path.open("r", encoding="utf-8") as file:
    for line in file:
        paper = json.loads(line)
        records.append({
            column: paper.get(column) for column in columns
        })

df = pd.DataFrame.from_records(records, columns=columns)

print(f"Loaded documents: {len(df):,}")
print(f"DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

display(df[["id", "title", "first_submission_date"]].head())

Loaded documents: 75,110
DataFrame memory: 112.4 MB


,id,title,first_submission_date
0,2101.00117,Multi-task Retrieval for Knowledge-Intensive T...,2021-01-01
1,2101.00121,WARP: Word-level Adversarial ReProgramming,2021-01-01
2,2101.00123,Intent Classification and Slot Filling for Pri...,2021-01-01
3,2101.00124,Discourse-level Relation Extraction via Graph ...,2021-01-01
4,2101.00130,Sensei: Self-Supervised Sensor Name Segmentation,2021-01-01


In [3]:

# Normalise whitespace without removing meaningful content
df["clean_title"] = (
    df["title"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["clean_abstract"] = (
    df["abstract"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Prepare submission dates for temporal analysis
df["first_submission_date"] = pd.to_datetime(
    df["first_submission_date"],
    errors="coerce"
)

df["year"] = df["first_submission_date"].dt.year

display(
    df[["id", "clean_title", "clean_abstract", "year"]].head()
)

,id,clean_title,clean_abstract,year
0,2101.00117,Multi-task Retrieval for Knowledge-Intensive T...,Retrieving relevant contexts from a large corp...,2021
1,2101.00121,WARP: Word-level Adversarial ReProgramming,Transfer learning from pretrained language mod...,2021
2,2101.00123,Intent Classification and Slot Filling for Pri...,Understanding privacy policies is crucial for ...,2021
3,2101.00124,Discourse-level Relation Extraction via Graph ...,The ability to capture complex linguistic stru...,2021
4,2101.00130,Sensei: Self-Supervised Sensor Name Segmentation,"A sensor name, typically an alphanumeric strin...",2021


In [4]:
# Verifing the dataset 

print("Total documents:", len(df))
print("Unique paper IDs:", df["id"].nunique())
print("Duplicate IDs:", df["id"].duplicated().sum())

print("Missing IDs:", df["id"].isna().sum())
print("Empty cleaned titles:", df["clean_title"].fillna("").eq("").sum())
print("Empty cleaned abstracts:", df["clean_abstract"].fillna("").eq("").sum())

print("Invalid dates:", df["first_submission_date"].isna().sum())
print("Outside 2021–2025:", (~df["year"].between(2021, 2025)).sum())

print(
    "Papers without cs.CL:",
    (~df["categories"].fillna("").str.split().map(
        lambda categories: "cs.CL" in categories
    )).sum()
)

print("\nDocuments by submission year:")
print(df["year"].value_counts().sort_index())

Total documents: 75110
Unique paper IDs: 75110
Duplicate IDs: 0
Missing IDs: 0
Empty cleaned titles: 0
Empty cleaned abstracts: 0
Invalid dates: 0
Outside 2021–2025: 0
Papers without cs.CL: 0

Documents by submission year:
year
2021     8083
2022     8971
2023    13603
2024    20689
2025    23764
Name: count, dtype: int64



## 2. Document representation

Titles and abstracts are concatenated to construct one textual document per paper.

This approach combines the concise topical information of titles with the richer descriptions provided by abstracts. Both fields receive equal treatment during concatenation, without introducing explicit title weighting.

The complete corpus will be used to construct a shared TF–IDF vocabulary, providing a consistent feature space for subsequent clustering and temporal comparisons.

In [7]:
# Combine cleaned titles and abstracts
df["document_text"] = (
    df["clean_title"] + " " + df["clean_abstract"]
)

# Count whitespace-separated words in each field
df["title_word_count"] = df["clean_title"].str.split().str.len()
df["abstract_word_count"] = df["clean_abstract"].str.split().str.len()

df["document_word_count"] = (
    df["title_word_count"] + df["abstract_word_count"]
)

# Estimate the proportion contributed by titles
df["title_word_share"] = (
    df["title_word_count"] / df["document_word_count"]
)

print("Documents:", len(df))
print("Empty combined documents:", df["document_text"].str.strip().eq("").sum())

print("\nAverage word counts:")
print(
    df[
        ["title_word_count", "abstract_word_count",
         "document_word_count", "title_word_share"]
    ].mean().round(3)
)

print("\nCombined document length distribution:")
print(
    df["document_word_count"].describe(
        percentiles=[0.01, 0.50, 0.99]
    ).round(1)
)

Documents: 75110
Empty combined documents: 0

Average word counts:
title_word_count        10.049
abstract_word_count    167.588
document_word_count    177.637
title_word_share         0.060
dtype: float64

Combined document length distribution:
count    75110.0
mean       177.6
std         43.3
min         20.0
1%          82.0
50%        175.0
99%        282.0
max        492.0
Name: document_word_count, dtype: float64


In [8]:

# Reproducible random sample of three papers
sample = df.sample(n=3, random_state=42)

for _, paper in sample.iterrows():
    print("=" * 80)
    print("Paper ID:", paper["id"])
    print("Year:", paper["year"])
    print("Title:", paper["clean_title"])
    print("Combined word count:", paper["document_word_count"])
    print("Title word share:", round(paper["title_word_share"], 3))
    print("\nCombined text (first 500 characters):")
    print(paper["document_text"][:500])
    print()

Paper ID: 2401.10580
Year: 2024
Title: PHOENIX: Open-Source Language Adaption for Direct Preference Optimization
Combined word count: 134
Title word share: 0.06

Combined text (first 500 characters):
PHOENIX: Open-Source Language Adaption for Direct Preference Optimization Large language models have gained immense importance in recent years and have demonstrated outstanding results in solving various tasks. However, despite these achievements, many questions remain unanswered in the context of large language models. Besides the optimal use of the models for inference and the alignment of the results to the desired specifications, the transfer of models to other languages is still an underdev

Paper ID: 2406.10984
Year: 2024
Title: Revisiting Cosine Similarity via Normalized ICA-transformed Embeddings
Combined word count: 134
Title word share: 0.052

Combined text (first 500 characters):
Revisiting Cosine Similarity via Normalized ICA-transformed Embeddings Cosine similarity is widely u

## 3. Tokenization and n-gram inspection.

In [9]:

from sklearn.feature_extraction.text import TfidfVectorizer

examples = {
    "Scientific terminology":
        "Large language models use retrieval-augmented generation "
        "and chain-of-thought reasoning.",

    "Scientific system terminology":
        "A dialogue system supports multilingual dialogue systems.",

    "Actual paper title":
        df.loc[
            df["id"] == "2401.10580", "clean_title"
        ].iloc[0]
}

for name, text in examples.items():
    print(f"\n{'=' * 70}")
    print(name)
    print("Original:", text)

    for ngram_range in [(1, 1), (1, 2), (1, 3)]:
        vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            stop_words="english"
        )

        tokens = vectorizer.build_analyzer()(text)

        print(f"\nN-gram range {ngram_range}:")
        print(tokens)



Scientific terminology
Original: Large language models use retrieval-augmented generation and chain-of-thought reasoning.

N-gram range (1, 1):
['large', 'language', 'models', 'use', 'retrieval', 'augmented', 'generation', 'chain', 'thought', 'reasoning']

N-gram range (1, 2):
['large', 'language', 'models', 'use', 'retrieval', 'augmented', 'generation', 'chain', 'thought', 'reasoning', 'large language', 'language models', 'models use', 'use retrieval', 'retrieval augmented', 'augmented generation', 'generation chain', 'chain thought', 'thought reasoning']

N-gram range (1, 3):
['large', 'language', 'models', 'use', 'retrieval', 'augmented', 'generation', 'chain', 'thought', 'reasoning', 'large language', 'language models', 'models use', 'use retrieval', 'retrieval augmented', 'augmented generation', 'generation chain', 'chain thought', 'thought reasoning', 'large language models', 'language models use', 'models use retrieval', 'use retrieval augmented', 'retrieval augmented generatio

## 4. TF–IDF feature extraction

In [10]:
import numpy as np
import sklearn
from time import perf_counter
from sklearn.feature_extraction.text import TfidfVectorizer

# Shared baseline configuration
tfidf_params = {
    "max_features": 10000,
    "min_df": 5,
    "max_df": 0.90,
    "stop_words": "english",
    "norm": "l2",
    "dtype": np.float32
}

# Candidate configurations
configurations = {
    "Unigrams": (1, 1),
    "Unigrams + bigrams": (1, 2)
}

vectorizers = {}
matrices = {}

print("scikit-learn version:", sklearn.__version__)

for name, ngram_range in configurations.items():

    vectorizer = TfidfVectorizer(
        **tfidf_params,
        ngram_range=ngram_range
    )

    start = perf_counter()

    X = vectorizer.fit_transform(df["document_text"])

    duration = perf_counter() - start

    vectorizers[name] = vectorizer
    matrices[name] = X

    print(f"\n{name}")
    print(f"Shape: {X.shape}")
    print(f"Nonzero elements: {X.nnz:,}")
    print(f"Fit time: {duration:.1f} seconds")


scikit-learn version: 1.9.1

Unigrams
Shape: (75110, 10000)
Nonzero elements: 6,207,305
Fit time: 4.1 seconds

Unigrams + bigrams
Shape: (75110, 10000)
Nonzero elements: 7,465,323
Fit time: 15.1 seconds


In [ ]:
results = []

for name, X in matrices.items():

    vectorizer = vectorizers[name]
    features = vectorizer.get_feature_names_out()

    unigram_count = sum(" " not in term for term in features)
    bigram_count = sum(" " in term for term in features)

    sparse_memory_mb = (
        X.data.nbytes
        + X.indices.nbytes
        + X.indptr.nbytes
    ) / (1024 ** 2)

    results.append({
        "Representation": name,
        "Documents": X.shape[0],
        "Features": X.shape[1],
        "Unigrams": unigram_count,
        "Bigrams": bigram_count,
        "Nonzero values": X.nnz,
        "Density (%)": 100 * X.nnz / (X.shape[0] * X.shape[1]),
        "Avg. features/document": X.nnz / X.shape[0],
        "Empty document vectors": (X.getnnz(axis=1) == 0).sum(),
        "Sparse matrix memory (MB)": sparse_memory_mb
    })

comparison = pd.DataFrame(results).set_index("Representation")

display(comparison.round(2))

,Documents,Features,Unigrams,Bigrams,Nonzero values,Density (%),Avg. features/document,Empty document vectors,Sparse matrix memory (MB)
Representation,,,,,,,,,
Unigrams,75110,10000,10000,0,6207305,0.83,82.64,0,47.64
Unigrams + bigrams,75110,10000,5654,4346,7465323,0.99,99.39,0,57.24


In [12]:
for name, X in matrices.items():

    vectorizer = vectorizers[name]
    features = vectorizer.get_feature_names_out()

    # Mean TF-IDF weight per feature over all documents
    mean_weights = np.asarray(X.mean(axis=0)).ravel()

    top_indices = np.argsort(mean_weights)[-20:][::-1]

    print(f"\n{'=' * 60}")
    print(f"TOP 20 FEATURES: {name}")

    for idx in top_indices:
        print(f"{features[idx]:30s} {mean_weights[idx]:.5f}")

    if name == "Unigrams + bigrams":

        bigram_indices = np.array([
            i for i, term in enumerate(features)
            if " " in term
        ])

        top_bigrams = bigram_indices[
            np.argsort(mean_weights[bigram_indices])[-20:][::-1]
        ]

        print("\nTOP 20 BIGRAMS")

        for idx in top_bigrams:
            print(f"{features[idx]:30s} {mean_weights[idx]:.5f}")
        


TOP 20 FEATURES: Unigrams
models                         0.03471
language                       0.03157
llms                           0.02794
model                          0.02723
data                           0.02423
text                           0.02041
reasoning                      0.02022
based                          0.01948
large                          0.01946
tasks                          0.01857
training                       0.01857
performance                    0.01841
knowledge                      0.01828
learning                       0.01782
task                           0.01681
llm                            0.01584
generation                     0.01512
human                          0.01465
speech                         0.01454
dataset                        0.01435

TOP 20 FEATURES: Unigrams + bigrams
models                         0.03197
language                       0.02924
llms                           0.02583
model                          0.02520


In [13]:
# Use the unigram + bigram representation
name = "Unigrams + bigrams"
X = matrices[name]
vectorizer = vectorizers[name]

features = vectorizer.get_feature_names_out()
vocabulary = vectorizer.vocabulary_

# Number of documents containing each feature
document_frequency = X.getnnz(axis=0)

# Terms selected for investigation
terms = [
    "models",
    "language",
    "reasoning",
    "llms",
    "language models",
    "natural language",
    "machine translation",
    "speech recognition",
    "question answering",
    "fine tuning",
    "github com",
    "available https",
    "state art"
]

rows = []

for term in terms:
    if term in vocabulary:
        idx = vocabulary[term]

        rows.append({
            "Term": term,
            "Document frequency": document_frequency[idx],
            "Document share (%)": (
                document_frequency[idx] / X.shape[0] * 100
            ),
            "IDF": vectorizer.idf_[idx]
        })
    else:
        print(f"Not in vocabulary: {term}")

term_diagnostics = pd.DataFrame(rows)

display(
    term_diagnostics.sort_values(
        "Document share (%)", ascending=False
    ).round(3)
)


,Term,Document frequency,Document share (%),IDF
0,models,55408,73.769,1.304
1,language,53359,71.041,1.342
4,language models,37132,49.437,1.704
3,llms,25821,34.378,2.068
12,state art,14305,19.045,2.658
2,reasoning,12107,16.119,2.825
5,natural language,11951,15.911,2.838
10,github com,9328,12.419,3.086
9,fine tuning,9007,11.992,3.121
11,available https,6232,8.297,3.489


In [14]:
paper_ids = [
    "2401.10580",
    "2406.10984",
    "2406.00755"
]

for paper_id in paper_ids:

    idx = df.index[df["id"] == paper_id][0]

    print(f"\n{'=' * 65}")
    print("Paper:", paper_id)
    print("Title:", df.loc[idx, "clean_title"])

    for name in configurations:

        X = matrices[name]
        features = vectorizers[name].get_feature_names_out()

        row = X.getrow(idx)
        order = np.argsort(row.data)[::-1][:12]

        print(f"\n{name} — Top 12 features:")

        for pos in order:
            feature_idx = row.indices[pos]
            weight = row.data[pos]

            print(f"{features[feature_idx]:30s} {weight:.4f}")



Paper: 2401.10580
Title: PHOENIX: Open-Source Language Adaption for Direct Preference Optimization

Unigrams — Top 12 features:
preference                     0.2161
direct                         0.2125
zephyr                         0.2080
remain                         0.2055
unanswered                     0.1972
underdeveloped                 0.1912
optimization                   0.1860
adaption                       0.1798
improvements                   0.1753
achievements                   0.1737
insights                       0.1720
immense                        0.1651

Unigrams + bigrams — Top 12 features:
direct preference              0.2491
preference optimization        0.2309
preference                     0.1882
direct                         0.1851
remain                         0.1790
optimization                   0.1620
models languages               0.1535
improvements                   0.1527
achievements                   0.1513
area research                  0.1

In [15]:
# Check vocabulary retention under the 10,000-feature limit
terms_to_check = [
    "ica",
    "zephyr",
    "cosine",
    "embeddings",
    "preference",
    "optimization",
    "mathematical reasoning",
    "cosine similarity",
    "direct preference",
    "language models"
]

rows = []

for term in terms_to_check:
    rows.append({
        "Term": term,
        "Unigrams": term in vectorizers["Unigrams"].vocabulary_,
        "Unigrams + bigrams": term in vectorizers["Unigrams + bigrams"].vocabulary_
    })

display(pd.DataFrame(rows))

# Examine prevalence of URL-like text
url_pattern = r"https?://|www\.|github\.com"

has_url = df["document_text"].str.contains(
    url_pattern,
    case=False,
    regex=True,
    na=False
)

print(f"\nDocuments containing URL-like text: {has_url.sum():,}")
print(f"Share of corpus: {has_url.mean():.2%}")


,Term,Unigrams,Unigrams + bigrams
0,ica,True,False
1,zephyr,True,False
2,cosine,True,True
3,embeddings,True,True
4,preference,True,True
5,optimization,True,True
6,mathematical reasoning,False,True
7,cosine similarity,False,True
8,direct preference,False,True
9,language models,False,True



Documents containing URL-like text: 11,296
Share of corpus: 15.04%



## 5. Targeted preprocessing refinement

The initial TF–IDF vocabulary contained several URL-related features, and approximately 15% of documents contained URL-like text.

To reduce non-topical vocabulary, a second text representation is constructed by removing explicit web addresses. The original document text is preserved for comparison.

No domain-specific scientific terminology is removed.

In [16]:
import re

# Match HTTP(S), www, and GitHub web addresses
url_pattern = r"(?i)\b(?:https?://|www\.|github\.com)\S*"

# Preserve the original document_text
df["document_text_no_urls"] = (
    df["document_text"]
    .str.replace(url_pattern, " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Validation
changed = df["document_text"] != df["document_text_no_urls"]

print("Documents modified:", changed.sum())
print("Share modified:", f"{changed.mean():.2%}")
print(
    "Empty documents after preprocessing:",
    df["document_text_no_urls"].eq("").sum()
)

# Inspect actual matched strings in two documents
for idx in df.index[changed][:2]:
    matches = re.findall(url_pattern, df.loc[idx, "document_text"])

    print("\nPaper:", df.loc[idx, "id"])
    print("Removed URL examples:", matches[:3])


Documents modified: 11295
Share modified: 15.04%
Empty documents after preprocessing: 0

Paper: 2101.00151
Removed URL examples: ['https://github.com/facebookresearch/DVDialogues.']

Paper: 2101.00154
Removed URL examples: ['https://github.com/HKUST-KnowComp/DISCOS-commonsense.']


### 5.1 Refit TF–IDF

In [17]:
# Keep all baseline parameters unchanged
vectorizer_clean = TfidfVectorizer(
    **tfidf_params,
    ngram_range=(1, 2)
)

X_clean = vectorizer_clean.fit_transform(
    df["document_text_no_urls"]
)

# Retain this new representation
name_clean = "Unigrams + bigrams (URL-cleaned)"

vectorizers[name_clean] = vectorizer_clean
matrices[name_clean] = X_clean

# Compare with the original combined representation
comparison_rows = []

for name in ["Unigrams + bigrams", name_clean]:
    X = matrices[name]
    features = vectorizers[name].get_feature_names_out()

    comparison_rows.append({
        "Representation": name,
        "Features": X.shape[1],
        "Unigrams": sum(" " not in t for t in features),
        "Bigrams": sum(" " in t for t in features),
        "Avg. active features": X.nnz / X.shape[0],
        "Empty vectors": (X.getnnz(axis=1) == 0).sum()
    })

display(pd.DataFrame(comparison_rows).round(2))

# Inspect selected vocabulary terms
terms_to_check = [
    "github com",
    "https github",
    "available https",
    "state art",
    "ica",
    "zephyr"
]

print("\nSelected vocabulary terms:")
for term in terms_to_check:
    old_present = term in vectorizers[
        "Unigrams + bigrams"
    ].vocabulary_
    new_present = term in vectorizer_clean.vocabulary_

    print(f"{term:20s} Before: {old_present}  After: {new_present}")

# Top 15 bigrams after URL removal
features = vectorizer_clean.get_feature_names_out()
mean_weights = np.asarray(X_clean.mean(axis=0)).ravel()

bigram_indices = np.array([
    i for i, term in enumerate(features) if " " in term
])

top_bigrams = bigram_indices[
    np.argsort(mean_weights[bigram_indices])[-15:][::-1]
]

print("\nTop 15 bigrams after URL removal:")
for idx in top_bigrams:
    print(f"{features[idx]:30s} {mean_weights[idx]:.5f}")


,Representation,Features,Unigrams,Bigrams,Avg. active features,Empty vectors
0,Unigrams + bigrams,10000,5654,4346,99.39,0
1,Unigrams + bigrams (URL-cleaned),10000,5647,4353,98.54,0



Selected vocabulary terms:
github com           Before: True  After: False
https github         Before: True  After: False
available https      Before: True  After: False
state art            Before: True  After: True
ica                  Before: False  After: False
zephyr               Before: False  After: False

Top 15 bigrams after URL removal:
language models                0.01812
large language                 0.01473
models llms                    0.00891
natural language               0.00839
fine tuning                    0.00830
state art                      0.00764
language model                 0.00676
pre trained                    0.00616
question answering             0.00532
real world                     0.00524
zero shot                      0.00505
language processing            0.00477
pre training                   0.00438
training data                  0.00413
low resource                   0.00413


## 6. Vocabulary size sensitivity analysis

In [18]:
# Expand vocabulary capacity while keeping other settings fixed
tfidf_params_20k = {
    **tfidf_params,
    "max_features": 20000
}

vectorizer_20k = TfidfVectorizer(
    **tfidf_params_20k,
    ngram_range=(1, 2)
)

start = perf_counter()

X_20k = vectorizer_20k.fit_transform(
    df["document_text_no_urls"]
)

duration = perf_counter() - start

# Compare both URL-cleaned representations
representations = {
    "10,000 features": (vectorizer_clean, X_clean),
    "20,000 features": (vectorizer_20k, X_20k)
}

rows = []

for name, (vectorizer, X) in representations.items():
    features = vectorizer.get_feature_names_out()

    sparse_memory = (
        X.data.nbytes
        + X.indices.nbytes
        + X.indptr.nbytes
    ) / (1024 ** 2)

    rows.append({
        "Representation": name,
        "Features": X.shape[1],
        "Unigrams": sum(" " not in t for t in features),
        "Bigrams": sum(" " in t for t in features),
        "Avg. active features": X.nnz / X.shape[0],
        "Density (%)": 100 * X.nnz / (X.shape[0] * X.shape[1]),
        "Memory (MiB)": sparse_memory,
        "Empty vectors": (X.getnnz(axis=1) == 0).sum()
    })

display(
    pd.DataFrame(rows).set_index("Representation").round(2)
)

print(f"\n20,000-feature fit time: {duration:.1f} seconds")

,Features,Unigrams,Bigrams,Avg. active features,Density (%),Memory (MiB),Empty vectors
Representation,,,,,,,
"10,000 features",10000,5647,4353,98.54,0.99,56.76,0
"20,000 features",20000,8094,11906,108.02,0.54,62.18,0



20,000-feature fit time: 16.1 seconds


In [19]:
# Compare retained vocabulary terms
vocab_10k = set(vectorizer_clean.get_feature_names_out())
vocab_20k = set(vectorizer_20k.get_feature_names_out())

added_terms = vocab_20k - vocab_10k
lost_terms = vocab_10k - vocab_20k

print("New features:", len(added_terms))
print("Previously retained features lost:", len(lost_terms))

print(
    "New unigrams:",
    sum(" " not in term for term in added_terms)
)
print(
    "New bigrams:",
    sum(" " in term for term in added_terms)
)

# Document frequency of newly admitted features
feature_df = X_20k.getnnz(axis=0)
vocab = vectorizer_20k.vocabulary_

new_unigrams = sorted(
    [t for t in added_terms if " " not in t],
    key=lambda t: feature_df[vocab[t]],
    reverse=True
)

new_bigrams = sorted(
    [t for t in added_terms if " " in t],
    key=lambda t: feature_df[vocab[t]],
    reverse=True
)

print("\nTop 15 newly retained unigrams:")
for term in new_unigrams[:15]:
    print(f"{term:30s} {feature_df[vocab[term]]:,}")

print("\nTop 15 newly retained bigrams:")
for term in new_bigrams[:15]:
    print(f"{term:30s} {feature_df[vocab[term]]:,}")

# Check specific scientific terms
print("\nSelected scientific terms:")

for term in ["ica", "zephyr", "cosine similarity",
             "mathematical reasoning", "direct preference"]:
    print(
        f"{term:30s}"
        f"10k: {term in vocab_10k}  "
        f"20k: {term in vocab_20k}"
    )

# Number of documents using at least one new feature
new_indices = [vocab[t] for t in added_terms]

additional_counts = X_20k[:, new_indices].getnnz(axis=1)

print(
    "\nDocuments with at least one additional feature:",
    f"{(additional_counts > 0).sum():,}"
)
print(
    "Average additional active features per document:",
    round(additional_counts.mean(), 2)
)


New features: 10000
Previously retained features lost: 0
New unigrams: 2447
New bigrams: 7553

Top 15 newly retained unigrams:
opened                         123
soon                           122
sought                         122
revolutionize                  122
handled                        121
conceptually                   121
06                             121
strengthens                    120
merges                         120
keeps                          120
04                             120
showcased                      120
08                             120
circumvent                     120
reformulate                    119

Top 15 newly retained bigrams:
garnered significant           123
provide depth                  123
various scenarios              123
methods work                   123
present simple                 123
making challenging             123
designed address               123
llms especially                123
maintaining competitive        122
a

In [20]:
example = (
    "A dialogue system uses chain-of-thought reasoning "
    "and state-of-the-art language models."
)

for stop_words in ["english", None]:

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words=stop_words
    )

    tokens = vectorizer.build_analyzer()(example)

    print(f"\nStop words: {stop_words}")
    print("Unigrams:", [t for t in tokens if " " not in t])
    print("Bigrams:", [t for t in tokens if " " in t])



Stop words: english
Unigrams: ['dialogue', 'uses', 'chain', 'thought', 'reasoning', 'state', 'art', 'language', 'models']
Bigrams: ['dialogue uses', 'uses chain', 'chain thought', 'thought reasoning', 'reasoning state', 'state art', 'art language', 'language models']

Stop words: None
Unigrams: ['dialogue', 'system', 'uses', 'chain', 'of', 'thought', 'reasoning', 'and', 'state', 'of', 'the', 'art', 'language', 'models']
Bigrams: ['dialogue system', 'system uses', 'uses chain', 'chain of', 'of thought', 'thought reasoning', 'reasoning and', 'and state', 'state of', 'of the', 'the art', 'art language', 'language models']


In [21]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Only change the stop-word setting
params_no_stop = {
    **tfidf_params,
    "stop_words": None
}

vectorizer_no_stop = TfidfVectorizer(
    **params_no_stop,
    ngram_range=(1, 2)
)

X_no_stop = vectorizer_no_stop.fit_transform(
    df["document_text_no_urls"]
)

for name, vectorizer, X in [
    ("English stop words", vectorizer_clean, X_clean),
    ("No stop-word removal", vectorizer_no_stop, X_no_stop)
]:
    features = vectorizer.get_feature_names_out()
    bigrams = [t for t in features if " " in t]

    stopword_bigrams = [
        t for t in bigrams
        if any(w in ENGLISH_STOP_WORDS for w in t.split())
    ]

    print(f"\n{name}")
    print("Vocabulary size:", len(features))
    print("Unigrams:", len(features) - len(bigrams))
    print("Bigrams:", len(bigrams))
    print("Bigrams containing stop words:", len(stopword_bigrams))
    print("Average active features:", round(X.nnz / X.shape[0], 2))

# Inspect prominent bigrams without stop-word removal
features = vectorizer_no_stop.get_feature_names_out()
mean_weights = np.asarray(X_no_stop.mean(axis=0)).ravel()

bigram_indices = np.array([
    i for i, term in enumerate(features) if " " in term
])

top_indices = bigram_indices[
    np.argsort(mean_weights[bigram_indices])[-20:][::-1]
]

print("\nTop 20 bigrams without stop-word removal:")
for idx in top_indices:
    print(f"{features[idx]:30s} {mean_weights[idx]:.5f}")

# Check selected meaningful and artificial expressions
terms_to_check = [
    "system",
    "dialogue system",
    "chain thought",
    "chain of",
    "of thought",
    "state art",
    "language models",
    "machine translation"
]

print("\nSelected vocabulary features:")
for term in terms_to_check:
    print(
        f"{term:25s}"
        f"English: {term in vectorizer_clean.vocabulary_}  "
        f"None: {term in vectorizer_no_stop.vocabulary_}"
    )



English stop words
Vocabulary size: 10000
Unigrams: 5647
Bigrams: 4353
Bigrams containing stop words: 0
Average active features: 98.54

No stop-word removal
Vocabulary size: 10000
Unigrams: 4174
Bigrams: 5826
Bigrams containing stop words: 4494
Average active features: 158.52

Top 20 bigrams without stop-word removal:
of the                         0.01671
language models                0.01653
large language                 0.01350
in the                         0.01218
in this                        0.01044
on the                         0.01002
we propose                     0.00927
to the                         0.00926
this paper                     0.00923
such as                        0.00852
show that                      0.00827
models llms                    0.00814
fine tuning                    0.00762
natural language               0.00761
based on                       0.00752
state of                       0.00716
we introduce                   0.00712
the art         